# Modern Document-Word Graph Builder Demo

This notebook demonstrates the usage of the refactored, modern implementation of the document-word heterogeneous graph builder.

## Key Improvements

1. **Object-Oriented Design**: Modular classes with clear responsibilities
2. **Type Hints**: Full type annotations for better IDE support and documentation
3. **Dataclasses**: Structured configuration and data containers
4. **Strategy Pattern**: Pluggable edge weight calculators
5. **Error Handling**: Proper validation and error messages
6. **Logging**: Comprehensive logging throughout the pipeline
7. **Documentation**: Comprehensive docstrings and comments

In [ ]:
import sys
import logging
from pathlib import Path

# Add project root to path
sys.path.append('/home/pwiesenbach/BertGCN')

from build_graph_refactored import (
    DocumentWordGraphBuilder, 
    GraphConfig, 
    PMICalculator, 
    TFIDFCalculator,
    setup_logging
)
from clinic_datasets import CleanClinicDataset
from transformers import AutoTokenizer

# Setup logging
setup_logging()
logging.info("Starting graph builder demonstration")

## Configuration

First, let's configure our graph builder with the desired parameters:

In [ ]:
# Create configuration
config = GraphConfig(
    window_size=20,              # Sliding window size for PMI calculation
    min_pmi_threshold=0.0,       # Minimum PMI value to include edge
    train_ratio=0.7,             # Proportion for training set
    val_ratio=0.1,               # Proportion for validation set (test = 1 - train - val)
    random_seed=42               # For reproducible results
)

print(f"Configuration: {config}")

## Initialize Components

Now let's initialize the tokenizer and graph builder:

In [ ]:
# Initialize tokenizer (replace with your preferred model)
model_name = "deepset/gbert-base"  # German BERT for clinical texts
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Create graph builder
builder = DocumentWordGraphBuilder(config, tokenizer)

print(f"Graph builder initialized with model: {model_name}")

## Load Dataset

Load your clinical dataset (this is just an example - replace with your actual dataset loading):

In [ ]:
# Example dataset loading (replace with your actual dataset)
# This would typically load a pre-processed clinical dataset

try:
    # Attempt to load existing dataset
    dataset_file = Path("data/medindcls_bert_letter.json")
    
    if dataset_file.exists():
        import pickle
        with open(dataset_file, "rb") as f:
            dataset = pickle.load(f)
        print(f"Loaded dataset with {len(dataset)} samples")
    else:
        # Create new dataset if file doesn't exist
        print("Dataset file not found. You would create your dataset here.")
        print("Example:")
        print("dataset = CleanClinicDataset(")
        print("    tokenizer=tokenizer,")
        print("    doclevel='letter',")
        print("    clean=True")
        print(")")
        dataset = None
        
except Exception as e:
    print(f"Error loading dataset: {e}")
    dataset = None

## Build Graph

If dataset is available, build the heterogeneous graph:

In [ ]:
if dataset is not None:
    # Build the graph
    graph_data = builder.build_graph(
        dataset=dataset,
        model_name=model_name,
        testunklar=False  # Set to True for special 'unklar' test split
    )
    
    print(f"Graph construction completed!")
    print(f"Adjacency matrix shape: {graph_data.adj.shape}")
    print(f"Number of edges: {graph_data.adj.nnz}")
    print(f"Training features shape: {graph_data.x.shape}")
    print(f"Training labels shape: {graph_data.y.shape}")
    
    # Save the graph
    save_path = Path("data/graphs")
    builder.save_graph(graph_data, save_path, "demo_graph")
    print(f"Graph saved to {save_path}")
    
else:
    print("Dataset not available - skipping graph construction")
    print("To use this with your data, ensure you have a properly formatted CleanClinicDataset")

## Individual Component Testing

Let's test the individual components with sample data:

In [ ]:
# Test PMI calculator with sample texts
sample_texts = [
    "patient diagnosed with diabetes mellitus type",
    "diabetes mellitus requires medication treatment", 
    "patient shows symptoms of diabetes",
    "medication treatment for diabetes mellitus",
    "type diabetes mellitus patient care"
]

# Build simple vocabulary
from collections import Counter
word_counter = Counter(word for text in sample_texts for word in text.split())
vocab = list(word_counter.keys())
word2id = {word: idx for idx, word in enumerate(vocab)}

print(f"Sample vocabulary: {vocab}")
print(f"Vocabulary size: {len(vocab)}")

# Test PMI calculation
pmi_calc = PMICalculator(config)
pmi_rows, pmi_cols, pmi_weights = pmi_calc.calculate_weights(
    texts=sample_texts,
    vocab=vocab,
    word2id=word2id,
    offset=10  # Arbitrary offset for demonstration
)

print(f"\nPMI Results:")
print(f"Number of word-word edges: {len(pmi_weights)}")
if pmi_weights:
    print(f"PMI weight range: [{min(pmi_weights):.3f}, {max(pmi_weights):.3f}]")
    
    # Show some examples
    for i in range(min(5, len(pmi_weights))):
        word_i = vocab[pmi_rows[i] - 10]  # Subtract offset
        word_j = vocab[pmi_cols[i] - 10]
        weight = pmi_weights[i]
        print(f"  {word_i} <-> {word_j}: {weight:.3f}")

## Architecture Overview

The refactored architecture provides several benefits:

In [ ]:
# Demonstrate the modular architecture
print("Refactored Architecture Benefits:")
print("\n1. Type Safety:")
print(f"   - GraphConfig ensures valid parameters: {type(config)}")
print(f"   - GraphData provides structured output: {type(graph_data) if 'graph_data' in locals() else 'GraphData'}")

print("\n2. Modularity:")
print(f"   - PMICalculator: {PMICalculator.__doc__}")
print(f"   - TFIDFCalculator: {TFIDFCalculator.__doc__}")

print("\n3. Extensibility:")
print("   - Easy to add new edge weight calculators")
print("   - Configuration-driven behavior")
print("   - Pluggable components")

print("\n4. Maintainability:")
print("   - Clear separation of concerns")
print("   - Comprehensive documentation")
print("   - Error handling and validation")

## Comparison with Original

Key improvements over the original implementation:

In [ ]:
improvements = {
    "Code Organization": {
        "Original": "Single large script with global functions",
        "Refactored": "Object-oriented with clear class hierarchy"
    },
    "Type Safety": {
        "Original": "No type hints, runtime errors",
        "Refactored": "Full type annotations, compile-time checking"
    },
    "Configuration": {
        "Original": "Hardcoded values scattered throughout",
        "Refactored": "Centralized GraphConfig with validation"
    },
    "Extensibility": {
        "Original": "Monolithic functions, hard to modify",
        "Refactored": "Strategy pattern, easy to extend"
    },
    "Error Handling": {
        "Original": "Minimal error checking",
        "Refactored": "Comprehensive validation and logging"
    },
    "Testing": {
        "Original": "Difficult to unit test",
        "Refactored": "Modular components, easy to test"
    }
}

for aspect, comparison in improvements.items():
    print(f"\n{aspect}:")
    print(f"  Original:   {comparison['Original']}")
    print(f"  Refactored: {comparison['Refactored']}")

## Usage Examples

Different ways to use the refactored system:

In [ ]:
# Example 1: Custom configuration
custom_config = GraphConfig(
    window_size=15,
    min_pmi_threshold=0.5,  # Only strong associations
    train_ratio=0.8,        # More training data
    val_ratio=0.1,
    random_seed=123
)

print("Example 1 - Custom Configuration:")
print(f"  Window size: {custom_config.window_size}")
print(f"  PMI threshold: {custom_config.min_pmi_threshold}")

# Example 2: Different edge weight calculators
print("\nExample 2 - Pluggable Components:")
print("  - Can easily swap PMI for other word association metrics")
print("  - Can use sklearn TfidfVectorizer instead of manual calculation")
print("  - Strategy pattern allows runtime component selection")

# Example 3: Error handling
try:
    invalid_config = GraphConfig(
        train_ratio=0.8,
        val_ratio=0.3  # This will cause train + val > 1.0
    )
except ValueError as e:
    print(f"\nExample 3 - Error Handling:")
    print(f"  Caught configuration error: {e}")

print("\nThe refactored system provides robust, maintainable, and extensible")
print("graph construction for heterogeneous document-word networks!")